> Os exemplos desta página usam dados sintéticos gerados pelo projeto. Não são observações ASTER nem permitem interpretar valores como reflectância, temperatura ou classes reais. Veja [os dados de exemplo](../../../../data/README.md).



# Capítulo 02 — Estatística de imagens

> **Referência:** Canty (2019), Cap. 2.  
> Teoria: `../aulas/random variables.md`, `../aulas/histogramas_e_covariancia.md`

Objetivo: tratar bandas como amostras de variáveis aleatórias; calcular média, desvio, histograma, covariância e correlação no raster sintético.

## 1. Intuição

O DN de um pixel é uma observação ruidosa e contextual. Agregando milhões de pixels, estimamos **média**, **variância** e a **estrutura de dependência** entre bandas (covariância). Sem isso, PCA, classificação gaussiana e muitos testes de mudança não têm base.

Dado: `data/exemplos/exemplo_sintetico.tif` (na raiz do repositório). Saídas: `../figuras/`.

## 2. Matemática

- Média e variância amostrais por banda.
- Histograma = distribuição empírica univariada.
- Covariância $\mathbf{S}$ ($N\times N$) e correlação $R_{jk}=S_{jk}/(s_j s_k)$.

Bandas ópticas próximas tendem a $R_{jk}$ alto → redundância (Cap. 3 remove isso via PCA).

## 3. Implementação

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_utils").is_dir())
DATA = REPO / "data/exemplos/exemplo_sintetico.tif"
OUT = REPO / "conteudo/analise-de-imagens/estatistica/figuras"
OUT.mkdir(parents=True, exist_ok=True)


def load_raster(path):
    try:
        import rasterio

        with rasterio.open(path) as src:
            return src.read().astype(np.float64)
    except Exception as e_rio:
        rio_err = e_rio
    try:
        from osgeo import gdal
        from osgeo.gdalconst import GA_ReadOnly

        gdal.AllRegister()
        ds = gdal.Open(str(path), GA_ReadOnly)
        if ds is None:
            raise RuntimeError(f"GDAL None: {path}")
        bands, rows, cols = ds.RasterCount, ds.RasterYSize, ds.RasterXSize
        data = np.empty((bands, rows, cols), dtype=np.float64)
        for b in range(bands):
            data[b] = ds.GetRasterBand(b + 1).ReadAsArray().astype(np.float64)
        ds = None
        return data
    except Exception as e_gdal:
        raise RuntimeError(
            f"Instale rasterio ou GDAL. rasterio={rio_err}; gdal={e_gdal}"
        )


image = load_raster(DATA)
nbands, nrows, ncols = image.shape
print(f"{DATA} → shape {image.shape}")

means = image.reshape(nbands, -1).mean(axis=1)
stds = image.reshape(nbands, -1).std(axis=1, ddof=1)
for b in range(nbands):
    print(f"Banda {b+1}: média={means[b]:.4f}, std={stds[b]:.4f}")

## 4. Histogramas e matrizes

In [ ]:
ncols_g = min(3, nbands)
nrows_g = int(np.ceil(nbands / ncols_g))
fig, axes = plt.subplots(nrows_g, ncols_g, figsize=(4 * ncols_g, 3 * nrows_g))
axes = np.atleast_1d(axes).ravel()
for b in range(nbands):
    axes[b].hist(image[b].ravel(), bins=64, color="steelblue", edgecolor="none")
    axes[b].set_title(f"Banda {b+1}")
for j in range(nbands, len(axes)):
    axes[j].set_axis_off()
fig.suptitle("Histogramas Exemplo sintético", y=1.01)
fig.tight_layout()
fig.savefig(OUT / "histogramas_bandas.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("Salvo histogramas_bandas.png")

G = image.reshape(nbands, -1)
S = np.cov(G, ddof=1)
std = np.sqrt(np.diag(S))
denom = np.outer(std, std)
R = np.divide(S, denom, out=np.zeros_like(S), where=denom > 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(S, cmap="viridis")
axes[0].set_title("Covariância")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(R, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_title("Correlação")
fig.colorbar(im1, ax=axes[1], fraction=0.046)
for ax in axes:
    ax.set_xlabel("banda")
    ax.set_ylabel("banda")
fig.tight_layout()
fig.savefig(OUT / "covariancia_correlacao.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("Salvo covariancia_correlacao.png")
print("Correlação:\n", np.round(R, 3))

## 5. Exercícios (CLI)

```bash
cd ../exercicios
python ex02_1.py                    # ou: python ex02_1.py /caminho/aster.tif
python ex02_2.py
```

Os scripts gravam PNGs em `../figuras/`.